In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [42]:
df = pd.read_csv("co2.csv")

In [43]:
df.head()

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


In [44]:
df.describe()

,Engine Size(L),Cylinders,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
count,7385.000000,7385.000000,7385.000000,7385.000000,7385.000000,7385.000000,7385.000000
mean,3.160068,5.615030,12.556534,9.041706,10.975071,27.481652,250.584699
std,1.354170,1.828307,3.500274,2.224456,2.892506,7.231879,58.512679
min,0.900000,3.000000,4.200000,4.000000,4.100000,11.000000,96.000000
25%,2.000000,4.000000,10.100000,7.500000,8.900000,22.000000,208.000000
50%,3.000000,6.000000,12.100000,8.700000,10.600000,27.000000,246.000000
75%,3.700000,6.000000,14.600000,10.200000,12.600000,32.000000,288.000000
max,8.400000,16.000000,30.600000,20.600000,26.100000,69.000000,522.000000


In [45]:
df.shape

(7385, 12)

In [46]:
df.dtypes

Make                                    str
Model                                   str
Vehicle Class                           str
Engine Size(L)                      float64
Cylinders                             int64
Transmission                            str
Fuel Type                               str
Fuel Consumption City (L/100 km)    float64
Fuel Consumption Hwy (L/100 km)     float64
Fuel Consumption Comb (L/100 km)    float64
Fuel Consumption Comb (mpg)           int64
CO2 Emissions(g/km)                   int64
dtype: object

In [47]:
df.nunique()

Make                                  42
Model                               2053
Vehicle Class                         16
Engine Size(L)                        51
Cylinders                              8
Transmission                          27
Fuel Type                              5
Fuel Consumption City (L/100 km)     211
Fuel Consumption Hwy (L/100 km)      143
Fuel Consumption Comb (L/100 km)     181
Fuel Consumption Comb (mpg)           54
CO2 Emissions(g/km)                  331
dtype: int64

In [48]:
df['Model'].value_counts()

Model
F-150 FFV                                 32
F-150 FFV 4X4                             32
MUSTANG                                   27
FOCUS FFV                                 24
SONIC                                     20
                                          ..
Highlander Hybrid AWD Limited/Platinum     1
RAV4 AWD TRD Off-Road                      1
Tacoma 4WD D-Cab TRD Off-Road/Pro          1
Atlas Cross Sport 4MOTION                  1
XC40 T4 AWD                                1
Name: count, Length: 2053, dtype: int64

In [49]:
df.columns

Index(['Make', 'Model', 'Vehicle Class', 'Engine Size(L)', 'Cylinders',
       'Transmission', 'Fuel Type', 'Fuel Consumption City (L/100 km)',
       'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)',
       'Fuel Consumption Comb (mpg)', 'CO2 Emissions(g/km)'],
      dtype='str')

In [51]:
df.shape

(7385, 12)

In [52]:
df_v1 = df.drop('Model', axis=1)
df_v1 = pd.get_dummies(df_v1, drop_first=True)
target_column = df['CO2 Emissions(g/km)']
df_v1 = df_v1 .drop('CO2 Emissions(g/km)', axis=1)
df_v1.shape

(7385, 92)

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_v1
y = target_column

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale only the X features
scaler = StandardScaler()

# Fit learns the mean/variance of training data, then transforms it
X_train_scaled = scaler.fit_transform(X_train)

# ONLY transform the test data using the training parameters
X_test_scaled = scaler.transform(X_test)

In [54]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print(f"Linear Regression RMSE Error: {lr_rmse:.4f}")

# Train Random Forest Model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print(f"Random Forest RMSE Error: {rf_rmse:.4f}")

Linear Regression RMSE Error: 5.6271
Random Forest RMSE Error: 4.1293


Trying with the Model column



In [55]:
df_v2 = df
df_v2 = df_v2.drop('CO2 Emissions(g/km)', axis=1)

In [60]:
# Method B: Map by Percentage (relative frequency)
text_cols = df_v2.select_dtypes(include=['str']).columns

for col in text_cols:
    percentage_map = df_v2[col].value_counts(normalize=True)
    df_v2[col] = df_v2[col].map(percentage_map)

df_v2.shape

(7385, 11)

In [61]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_v2
y = target_column

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale only the X features
scaler = StandardScaler()

# Fit learns the mean/variance of training data, then transforms it
X_train_scaled = scaler.fit_transform(X_train)

# ONLY transform the test data using the training parameters
X_test_scaled = scaler.transform(X_test)

In [62]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print(f"Linear Regression RMSE Error: {lr_rmse:.4f}")

# Train Random Forest Model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print(f"Random Forest RMSE Error: {rf_rmse:.4f}")

Linear Regression RMSE Error: 16.1073
Random Forest RMSE Error: 2.8773


combine

In [64]:
df_v3 = df
df_v3 = df_v3.drop('CO2 Emissions(g/km)', axis=1)

In [65]:
# Method B: Map by Percentage (relative frequency)
percentage_map = df_v3['Model'].value_counts(normalize=True)
df_v3['Model'] = df_v3['Model'].map(percentage_map)
df_v3 = pd.get_dummies(dataframe, drop_first=True)
df_v3.shape

(7385, 94)

In [66]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_v3
y = target_column

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale only the X features
scaler = StandardScaler()

# Fit learns the mean/variance of training data, then transforms it
X_train_scaled = scaler.fit_transform(X_train)

# ONLY transform the test data using the training parameters
X_test_scaled = scaler.transform(X_test)

In [67]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print(f"Linear Regression RMSE Error: {lr_rmse:.4f}")

# Train Random Forest Model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print(f"Random Forest RMSE Error: {rf_rmse:.4f}")

Linear Regression RMSE Error: 0.0000
Random Forest RMSE Error: 0.1067


Deep learning model